In [ ]:
"""
BirdCLEF+ 2026 — Inference Notebook (CPU, < 90 min)
SED with attention head, adaptive ensemble, time-shift TTA, site/hour priors.
Adapts model count and TTA based on test set size to fit within time budget.
"""
import os
import time
import subprocess
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import librosa
from pathlib import Path
from tqdm import tqdm

start_time = time.time()

In [ ]:
# ── Auto-detect paths ─────────────────────────────────────
COMP_DATA = Path("/kaggle/input/birdclef-2026")
MODEL_DIR = Path("/kaggle/input/birdclef2026-model-weights")
OUTPUT_DIR = Path("/kaggle/working")

if not (COMP_DATA / "taxonomy.csv").exists():
    result = subprocess.run(["find", "/kaggle/input", "-name", "taxonomy.csv", "-type", "f"],
                            capture_output=True, text=True)
    found = result.stdout.strip().split("\n")
    if found and found[0]:
        COMP_DATA = Path(found[0]).parent
        print(f"Using detected path: {COMP_DATA}")

if not list(MODEL_DIR.glob("exp*_*.pth")):
    result = subprocess.run(["find", "/kaggle/input", "-name", "exp*_*.pth", "-type", "f"],
                            capture_output=True, text=True)
    found = result.stdout.strip().split("\n")
    if found and found[0]:
        MODEL_DIR = Path(found[0]).parent
        print(f"Using detected model dir: {MODEL_DIR}")

In [ ]:
# ── Config (must match training) ───────────────────────────
SR = 32000
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
FMIN = 0
FMAX = 16000
IMG_SIZE = 224
BACKBONE = "tf_efficientnet_b0.ns_jft_in1k"
TIME_BUDGET_SEC = 80 * 60  # 80 min budget (10 min margin)

In [ ]:
# ── Taxonomy ───────────────────────────────────────────────
taxonomy_df = pd.read_csv(COMP_DATA / "taxonomy.csv")
SPECIES_LIST = sorted(taxonomy_df["primary_label"].astype(str).tolist())
SPECIES2IDX = {sp: i for i, sp in enumerate(SPECIES_LIST)}
NUM_CLASSES = len(SPECIES_LIST)
print(f"Number of classes: {NUM_CLASSES}")

In [ ]:
# ── Sample submission ──────────────────────────────────────
sample_sub = pd.read_csv(COMP_DATA / "sample_submission.csv")
species_cols = [c for c in sample_sub.columns if c != "row_id"]

In [ ]:
# ── Build site/hour priors from labeled soundscapes ────────
FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_filename(name):
    m = FNAME_RE.match(name)
    if not m:
        return None, -1
    _, site, _, hms = m.groups()
    return site, int(hms[:2])

prior_tables = None
try:
    ss_labels = pd.read_csv(COMP_DATA / "train_soundscapes_labels.csv").drop_duplicates()
    site_species = {}
    hour_species = {}
    for _, row in ss_labels.iterrows():
        site, hour = parse_filename(row["filename"])
        if site is None:
            continue
        species = [s.strip() for s in str(row["primary_label"]).split(";")]
        site_species.setdefault(site, set()).update(species)
        hour_species.setdefault(hour, set()).update(species)
    prior_tables = {"site": site_species, "hour": hour_species}
    print(f"Built priors: {len(site_species)} sites, {len(hour_species)} hours")
except Exception as e:
    print(f"Could not build priors: {e}")


def apply_prior(preds, site, hour, strength=0.3):
    if prior_tables is None:
        return preds
    boosted = preds.copy()
    known = set()
    if site in prior_tables["site"]:
        known |= prior_tables["site"][site]
    if hour in prior_tables["hour"]:
        known |= prior_tables["hour"][hour]
    for sp in known:
        if sp in SPECIES2IDX:
            idx = SPECIES2IDX[sp]
            boosted[:, idx] = np.maximum(boosted[:, idx], boosted[:, idx] + strength * (1 - boosted[:, idx]))
    return boosted

In [ ]:
# ── Model — SED with attention head ───────────────────────
class AttentionHead(nn.Module):
    def __init__(self, in_features, num_classes, dropout=0.3):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(in_features, in_features),
            nn.Tanh(),
            nn.Linear(in_features, num_classes),
            nn.Softmax(dim=1),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes),
        )

    def forward(self, x):
        framewise = self.classifier(x)
        attn_weights = self.attention(x)
        clipwise = (framewise * attn_weights).sum(dim=1)
        return clipwise


class BirdSEDModel(nn.Module):
    def __init__(self, backbone_name, num_classes):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=False,
            in_chans=1, num_classes=0, global_pool="",
        )
        with torch.no_grad():
            dummy = torch.randn(1, 1, IMG_SIZE, IMG_SIZE)
            feat = self.backbone(dummy)
            self.feat_dim = feat.shape[1]
        self.head = AttentionHead(self.feat_dim, num_classes, dropout=0.3)

    def forward(self, x):
        feat = self.backbone(x)
        feat = feat.mean(dim=2).permute(0, 2, 1)
        return self.head(feat)

In [ ]:
# ── Load ensemble models ─────────────────────────────────
def load_ensemble(model_dir, max_models=6):
    """Load diverse ensemble: exp9 > exp8 > exp7 > exp3 (priority order)."""
    models = []
    for prefix in ["exp9_aug", "exp8_focal", "exp7_st2", "exp3_sed"]:
        found = sorted(model_dir.glob(f"{prefix}_fold*_best.pth"))
        for ckpt_path in found:
            if len(models) >= max_models:
                break
            print(f"  Loading {ckpt_path.name}...")
            model = BirdSEDModel(BACKBONE, NUM_CLASSES)
            state = torch.load(ckpt_path, map_location="cpu", weights_only=False)
            model.load_state_dict(state["model_state_dict"])
            model.eval()
            models.append(model)
        if len(models) >= max_models:
            break
    return models

In [ ]:
# ── Adaptive configuration based on test set size ──────────
test_dir = COMP_DATA / "test_soundscapes"
test_files = sorted(test_dir.glob("*.ogg"))
n_test = len(test_files)
print(f"Found {n_test} test soundscape files")

# Estimate time per forward pass on CPU (~80ms for B0@224)
MS_PER_PASS = 80
N_SEGMENTS = 12

# Calculate how many models × TTA we can afford
# total_passes = n_test × N_SEGMENTS × n_models × n_tta
# total_time_ms = total_passes × MS_PER_PASS
budget_ms = TIME_BUDGET_SEC * 1000
# Reserve 5 min for loading + audio I/O
budget_ms -= 5 * 60 * 1000

if n_test > 0:
    passes_per_file = budget_ms / (n_test * MS_PER_PASS)
    # passes_per_file = N_SEGMENTS × n_models × n_tta
    max_model_tta = passes_per_file / N_SEGMENTS

    # Try configurations in preference order
    configs = [
        (6, 3),  # 6 models, 3 TTA = 18 per segment
        (6, 2),  # 6 models, 2 TTA = 12
        (4, 2),  # 4 models, 2 TTA = 8
        (2, 3),  # 2 models, 3 TTA = 6
        (2, 2),  # 2 models, 2 TTA = 4
        (2, 1),  # 2 models, 1 TTA = 2
        (1, 1),  # 1 model, 1 TTA = 1
    ]
    chosen = (2, 1)  # fallback
    for nm, nt in configs:
        if nm * nt <= max_model_tta:
            chosen = (nm, nt)
            break

    n_models_target, n_tta = chosen
else:
    n_models_target, n_tta = 6, 2

# Build TTA shifts
if n_tta >= 3:
    TTA_SHIFTS = [0, int(SR * 1.5), int(SR * 3.0)]
elif n_tta >= 2:
    TTA_SHIFTS = [0, int(SR * 2.5)]
else:
    TTA_SHIFTS = [0]

models = load_ensemble(MODEL_DIR, max_models=n_models_target)
print(f"Ensemble: {len(models)} models, {len(TTA_SHIFTS)} TTA shifts")
est_time = n_test * N_SEGMENTS * len(models) * len(TTA_SHIFTS) * MS_PER_PASS / 1000 / 60
print(f"Estimated inference time: {est_time:.1f} min")

In [ ]:
# ── Audio processing ───────────────────────────────────────
def compute_mel(wav, sr=SR):
    mel = librosa.feature.melspectrogram(
        y=wav, sr=sr,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX,
        power=2.0,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    return mel_db.astype(np.float32)


def process_soundscape(filepath, models):
    """Process a 1-minute soundscape with TTA and ensemble."""
    wav, _ = librosa.load(filepath, sr=SR)
    target_len = int(SR * DURATION)

    all_preds = np.zeros((N_SEGMENTS, NUM_CLASSES), dtype=np.float32)

    for seg in range(N_SEGMENTS):
        seg_preds = []
        for shift in TTA_SHIFTS:
            start = seg * target_len + shift
            end = start + target_len
            chunk = wav[max(0, start):end]
            if len(chunk) < target_len:
                chunk = np.pad(chunk, (0, target_len - len(chunk)))
            chunk = chunk[:target_len]

            mel = compute_mel(chunk)
            mel_t = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            mel_t = F.interpolate(mel_t, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)

            with torch.no_grad():
                for model in models:
                    logits = model(mel_t)
                    prob = torch.sigmoid(logits).numpy()[0]
                    seg_preds.append(prob)

        all_preds[seg] = np.mean(seg_preds, axis=0)

    return all_preds

In [ ]:
# ── Main inference loop ────────────────────────────────────
rows = []
for fpath in tqdm(test_files, desc="Inference"):
    fname = fpath.stem
    site, hour = parse_filename(fpath.name)
    preds = process_soundscape(fpath, models)
    preds = apply_prior(preds, site, hour)

    for seg_idx in range(N_SEGMENTS):
        end_sec = (seg_idx + 1) * 5
        row_id = f"{fname}_{end_sec}"
        row = {"row_id": row_id}
        for i, sp in enumerate(species_cols):
            sp_idx = SPECIES2IDX.get(sp, -1)
            if sp_idx >= 0:
                row[sp] = float(preds[seg_idx, sp_idx])
            else:
                row[sp] = 1.0 / NUM_CLASSES
        rows.append(row)

In [ ]:
# ── Build submission ───────────────────────────────────────
if rows:
    submission = pd.DataFrame(rows)
    submission = submission[sample_sub.columns]
else:
    submission = sample_sub.copy()

submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)

elapsed = time.time() - start_time
print(f"\nSubmission saved: {len(submission)} rows, {elapsed/60:.1f} min elapsed")
print(f"Models in ensemble: {len(models)}, TTA shifts: {len(TTA_SHIFTS)}")
print(submission.head())